# 03 · Synthetic route-hour ridership scenario

**Purpose:** create a transparent route-hour *scenario* when route-wise passenger counts are unavailable. The scenario allocates an aggregate daily passenger total in proportion to scheduled trip departures, with a simple peak-hour weighting.

**This is not observed MTC ridership and must not be reported as measured data.** GTFS has no route-wise passenger labels, so this dataset cannot train or validate a supervised “overcrowded / balanced” classifier. Use it for a demonstration of the pipeline and sensitivity analysis only. Replace it with MTC route-level counts if you obtain them.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.gtfs import resolve_gtfs_dir, load_gtfs

GTFS_DIR = resolve_gtfs_dir(PROJECT_ROOT)
feed = load_gtfs(GTFS_DIR)
print(f"Feed folder: {GTFS_DIR}")
from src.gtfs import active_service_ids, first_trip_departures

SERVICE_DATE = "2026-10-26"  # Must be active in calendar.txt
# MTC performance page reported 32.24 lakh passengers/day for FY 2024-25.
# This is an all-network average used only as an illustrative calibration target.
# Edit this number and the year/source below if your mentor selects a different baseline.
REFERENCE_DAILY_PASSENGERS = 3_224_000
REFERENCE_SOURCE = "MTC Performance Indicators, FY 2024-25; aggregate Chennai network daily average"
REFERENCE_URL = "https://mtcbus.tn.gov.in/Home/performance"

# Peak-hour scenario multipliers (assumptions, not learned from passenger data).
PEAK_HOURS = {7, 8, 9, 17, 18, 19}
PEAK_MULTIPLIER = 1.6
OFF_PEAK_MULTIPLIER = 1.0


## 1. Build scheduled route-hour supply for the selected date

In [ ]:
active_ids = active_service_ids(feed["calendar"], SERVICE_DATE)
if not active_ids:
    raise ValueError("No calendar service active on SERVICE_DATE. Choose a covered operating date.")

departures = first_trip_departures(feed["stop_times"])
trips = feed["trips"].copy()
trips["service_id"] = trips["service_id"].astype(str)
trips = trips[trips["service_id"].isin(active_ids)].merge(departures, on="trip_id", how="left", validate="one_to_one")
trips = trips.merge(feed["routes"], on="route_id", how="left", validate="many_to_one").dropna(subset=["departure_hour"])
trips["departure_hour"] = trips["departure_hour"].astype(int)
route_hour = (trips.groupby(["route_id", "route_short_name", "route_long_name", "departure_hour"], dropna=False)
              .size().rename("scheduled_trip_count").reset_index())
if route_hour.empty:
    raise ValueError("No trips with a valid first-stop departure time were found.")
display(route_hour.head())

## 2. Allocate the aggregate total, preserving the total exactly

In [ ]:
route_hour["scenario_multiplier"] = route_hour["departure_hour"].map(
    lambda hour: PEAK_MULTIPLIER if hour in PEAK_HOURS else OFF_PEAK_MULTIPLIER
)
route_hour["allocation_weight"] = route_hour["scheduled_trip_count"] * route_hour["scenario_multiplier"]
route_hour["unrounded_estimate"] = (
    REFERENCE_DAILY_PASSENGERS * route_hour["allocation_weight"] / route_hour["allocation_weight"].sum()
)
# Largest-remainder rounding: integer counts that sum exactly to the selected daily total.
route_hour["synthetic_ridership_estimate"] = route_hour["unrounded_estimate"].astype(int)
remainder = int(REFERENCE_DAILY_PASSENGERS - route_hour["synthetic_ridership_estimate"].sum())
if remainder > 0:
    top_fractional = (route_hour["unrounded_estimate"] - route_hour["synthetic_ridership_estimate"]).nlargest(remainder).index
    route_hour.loc[top_fractional, "synthetic_ridership_estimate"] += 1
route_hour["data_status"] = "synthetic_scenario_not_observed"
route_hour["reference_daily_total"] = REFERENCE_DAILY_PASSENGERS
route_hour["reference_source"] = REFERENCE_SOURCE

assert int(route_hour["synthetic_ridership_estimate"].sum()) == REFERENCE_DAILY_PASSENGERS
print("Estimated total:", f"{route_hour['synthetic_ridership_estimate'].sum():,}")
print("Calibration total:", f"{REFERENCE_DAILY_PASSENGERS:,}")
display(route_hour.head(10))

## 3. Inspect route totals and the synthetic hourly profile

In [ ]:
route_estimates = (route_hour.groupby(["route_id", "route_short_name", "route_long_name"], dropna=False)
                   .agg(synthetic_daily_estimate=("synthetic_ridership_estimate", "sum"),
                        scheduled_trips=("scheduled_trip_count", "sum"))
                   .sort_values("synthetic_daily_estimate", ascending=False).reset_index())
display(route_estimates.head(20))

hour_profile = route_hour.groupby("departure_hour")["synthetic_ridership_estimate"].sum().reindex(range(24), fill_value=0)
ax = hour_profile.plot(kind="bar", figsize=(12, 4), color="#f18f01", title="Synthetic daily ridership allocation by scheduled departure hour")
ax.set_xlabel("Departure hour (local feed time)")
ax.set_ylabel("Synthetic passenger estimate")
ax.grid(axis="y", alpha=.25)


## 4. Save with provenance labels

In [ ]:
output_dir = PROJECT_ROOT / "dataset" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"synthetic_route_hour_ridership_{SERVICE_DATE}.csv"
route_hour.drop(columns=["unrounded_estimate"]).to_csv(output_path, index=False)
print(f"Wrote {output_path}")
print("Keep this file labeled synthetic in every chart, model, and report.")

## Model limitations and next steps

1. The daily total is a network-wide calibration target; it does not reveal how riders are split between routes.
2. Allocating in proportion to scheduled trips assumes each trip has similar demand before time-of-day multipliers. That assumption is unverified.
3. The multiplier is illustrative. Test alternatives (for example, all hours equal; moderate peaks; stronger peaks) and compare how rankings change.
4. “Route-hour ridership” here means an allocated count attached to the trip's first-stop departure hour. It is not stop-level boarding, load on a segment, or passengers simultaneously on a bus.
5. Without vehicle capacity and observed passenger loads, do not claim a route is overcrowded. The project can report **scenario-based demand pressure** only, clearly defined by the selected assumptions.
6. For stronger estimates, collect route/time-band counts with a small field survey or ask MTC/CUMTA for anonymized ticketing counts. Record date, route, direction, time band, count definition, source, and sampling method. Validate the synthetic scenario against real counts before using predictive labels.